# Dataset

What HalOmi holds, before anything is asked of it. The label balance differs
sharply by direction, which is the first thing to know: a judge that always says
no hallucination scores well on English to Russian and badly on Manipuri to
English, and a metric that does not account for that would mislead.

In [9]:
import json
import sys
from pathlib import Path
import pandas as pd

if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [10]:
%load_ext autoreload
%autoreload 2

import backends
import evaluate
import prompts as templates
import run
import settings
import utils

utils.make_directories()
pd.set_option('display.max_colwidth', 90)
# Built from the dataset rather than shipped, so that what the pipeline reads is
# always derived from what is in data/halomi rather than from a stale copy.
if not settings.ITEMS_PATH.exists():
    raise SystemExit(
        'Nothing has been built yet. From the repository root, run:\n'
        '    python scripts/build.py\n'
        'That writes data/benchmark/items.csv and prompts.csv, which every '
        'notebook and stage reads.')

print('Ready')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Ready


In [11]:
items = utils.read_table(settings.ITEMS_PATH)
print(f'{len(items):,} items across {items["direction"].nunique()} directions')
display(items.head(3)[['item_id', 'direction', 'grade', 'answer']])

720 items across 18 directions


,item_id,direction,grade,answer
0,arbeng-0003,Arabic to English,1_No_hallucination,No hallucination
1,arbeng-0008,Arabic to English,1_No_hallucination,No hallucination
2,arbeng-0013,Arabic to English,1_No_hallucination,No hallucination


## Label by Direction

In [12]:
balance = (items.assign(positive=items['answer'] == settings.HALLUCINATION)
           .groupby('direction')
           .agg(items=('item_id', 'size'), hallucinated=('positive', 'mean')))
balance['hallucinated'] = balance['hallucinated'].map('{:.0%}'.format)
display(balance.sort_values('items', ascending=False))

,items,hallucinated
direction,,
Arabic to English,40,15%
Chinese to English,40,20%
Yoruba to English,40,15%
Spanish to Yoruba,40,35%
Spanish to English,40,20%
Russian to English,40,8%
Manipuri to English,40,72%
Kashmiri to English,40,40%
German to English,40,10%


## Severity

The task is binary, but HalOmi grades hallucination in four levels. A judge that
catches full hallucinations and misses small ones is not the same as one that
catches neither, and the binary label hides that.

In [13]:
display(items['grade'].value_counts().to_frame('items'))

,items
grade,
1_No_hallucination,541
3_Partial_hallucination,65
4_Full_hallucination,62
2_Small_hallucination,52


## Text

In [14]:
for row in items.sample(3, random_state=1).to_dict('records'):
    print(f"{row['direction']}  ({row['answer']})")
    print(f"  source      {row['source_text'][:110]}")
    print(f"  translation {row['target_text'][:110]}")
    print()

German to English  (Hallucination)
  source      Hallo Triptychon,
  translation Hey, Triptychon, what are you doing?

German to English  (No hallucination)
  source      Die antiken Einrichtungen, das Nichtvorhandensein moderner Bequemlichkeiten und eine gewisse elegante Bejahrth
  translation The antique facilities, the absence of modern amenities and a certain elegance of age also make up its charact

English to Manipuri  (Hallucination)
  source      suk my dikkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
  translation ঐহাক্না ঐহাকপু থাগৎচরি, ঐহাক্না নহাকপু থুগৎচরি। ঐহাক্না অদোমগী মফমদা থাগৎকদবনি। ঐহাক্কী মফমদা হরাও-হরাও তৌবিয়

